# Catalog Feature Audit: Product Creation

**Author:** shazeb.asad | **Snapshot Date:** April 14, 2026

**Source:** [Confluence](https://getyourguide.atlassian.net/wiki/spaces/DA/pages/4206461002/Catalog+Feature+Audit+-+Product+Creation)

**Tables used:**
- `production.supply_analytics.tour_creation_funnel` — Primary source; one row per (tour_id, step_name). Key fields: `tour_id`, `step_name`, `step_number`, `first_timestamp`. Covers wizard steps from `Activity Creation Start` (step 1) through `Activity Submitted` (step 28). Data available from 2024-01-01 to present.
- `production.db_mirror_dbz.catalog__tour_generated_content` — AI content generation requests. One row per generation attempt. Key fields: `tour_id`, `supplier_id`, `content_generation_status` (completed / failed), `update_timestamp`.
- `production.db_mirror_dbz.catalog__tour_business_category` — Business category assignments per tour. Key fields: `tour_id`, `level1`, `level2`, `level3`.
- `production.supply.dim_activity_lookback_activation` — Rolling 28-day booking lookback per tour per date. Key fields: `tour_id`, `date`, `has_booking_in_preceding_28_days`. Partitioned by `date`.

**Key definitions:**
- **Tour creation start:** A supplier session that reached the `Activity Creation Start` funnel step. Primary denominator for funnel and adoption metrics.
- **Tour submission:** A creation session that reached the `Activity Submitted` funnel step.
- **AI path:** Tours that reached the `AI Content Creator` step (step 3) in `tour_creation_funnel`.
- **Manual path:** Tours that reached `Activity Creation Start` but not `AI Content Creator`.

**Analysis window:** 2026-01-01 — 2026-04-14 (current); 2025-01-01 — 2025-04-14 (2025 baseline for YoY comparisons)

In [ ]:
SNAPSHOT_DATE    = "2026-04-14"
ANALYSIS_START   = "2026-01-01"
ANALYSIS_END     = "2026-04-15"   # exclusive upper bound

BASELINE_START   = "2025-01-01"
BASELINE_END     = "2025-04-15"   # exclusive upper bound; same calendar window in prior year

FUNNEL_START     = "2026-01-15"   # Jan 15 – Apr 14 window used for wizard step drop-off analysis (Q3)
FUNNEL_END       = "2026-04-15"

---
## Q1: What is the overall creation volume and trend?

In [ ]:
# Table 1.1 — Monthly Creation Volume: 2025 vs 2026 (Jan–Apr, same calendar window)
df_q1_1 = spark.sql(f"""
WITH starts AS (
  SELECT
    tour_id,
    first_timestamp,
    YEAR(first_timestamp)                         AS yr,
    MONTH(first_timestamp)                        AS mth_num,
    DATE_FORMAT(first_timestamp, 'MMM')           AS mth_name
  FROM production.supply_analytics.tour_creation_funnel
  WHERE step_name = 'Activity Creation Start'
    AND (
      (first_timestamp >= '{BASELINE_START}' AND first_timestamp < '{BASELINE_END}')
      OR (first_timestamp >= '{ANALYSIS_START}' AND first_timestamp < '{ANALYSIS_END}')
    )
),
submissions AS (
  SELECT DISTINCT tour_id
  FROM production.supply_analytics.tour_creation_funnel
  WHERE step_name = 'Activity Submitted'
)
SELECT
  s.yr                                                                                                            AS year,
  s.mth_name                                                                                                     AS month,
  COUNT(DISTINCT s.tour_id)                                                                                      AS tours_started,
  COUNT(DISTINCT sub.tour_id)                                                                                    AS tours_submitted,
  CONCAT(ROUND(100.0 * COUNT(DISTINCT sub.tour_id) / NULLIF(COUNT(DISTINCT s.tour_id), 0), 1), '%')            AS submission_rate
FROM starts s
LEFT JOIN submissions sub ON s.tour_id = sub.tour_id
GROUP BY s.yr, s.mth_num, s.mth_name
ORDER BY s.yr, s.mth_num
""")

display(df_q1_1)

In [ ]:
# Table 1.2 — YoY Summary: Jan–Apr 2025 vs Jan–Apr 2026
df_q1_2 = spark.sql(f"""
WITH starts AS (
  SELECT
    tour_id,
    first_timestamp,
    YEAR(first_timestamp) AS yr
  FROM production.supply_analytics.tour_creation_funnel
  WHERE step_name = 'Activity Creation Start'
    AND (
      (first_timestamp >= '{BASELINE_START}' AND first_timestamp < '{BASELINE_END}')
      OR (first_timestamp >= '{ANALYSIS_START}' AND first_timestamp < '{ANALYSIS_END}')
    )
),
submissions AS (
  SELECT DISTINCT tour_id
  FROM production.supply_analytics.tour_creation_funnel
  WHERE step_name = 'Activity Submitted'
)
SELECT
  s.yr                                                                                                            AS year,
  COUNT(DISTINCT s.tour_id)                                                                                      AS tours_started,
  COUNT(DISTINCT sub.tour_id)                                                                                    AS tours_submitted,
  CONCAT(ROUND(100.0 * COUNT(DISTINCT sub.tour_id) / NULLIF(COUNT(DISTINCT s.tour_id), 0), 1), '%')            AS submission_rate
FROM starts s
LEFT JOIN submissions sub ON s.tour_id = sub.tour_id
GROUP BY s.yr
ORDER BY s.yr
""")

display(df_q1_2)

---
## Q2: How widely is the AI Content Creator being used?

In [ ]:
# Table 2.1 — AI vs Manual Submission Rate (2026 analysis window)
df_q2_1 = spark.sql(f"""
WITH starts AS (
  SELECT tour_id, first_timestamp
  FROM production.supply_analytics.tour_creation_funnel
  WHERE step_name = 'Activity Creation Start'
    AND first_timestamp >= '{ANALYSIS_START}'
    AND first_timestamp <  '{ANALYSIS_END}'
),
ai_path AS (
  SELECT DISTINCT tour_id
  FROM production.supply_analytics.tour_creation_funnel
  WHERE step_name = 'AI Content Creator'
),
submissions AS (
  SELECT DISTINCT tour_id
  FROM production.supply_analytics.tour_creation_funnel
  WHERE step_name = 'Activity Submitted'
)
SELECT
  CASE WHEN ai.tour_id IS NOT NULL THEN 'AI Content Creator' ELSE 'Manual (No AI)' END  AS creation_path,
  COUNT(DISTINCT s.tour_id)                                                               AS tours_started,
  COUNT(DISTINCT sub.tour_id)                                                             AS tours_submitted,
  CONCAT(ROUND(100.0 * COUNT(DISTINCT sub.tour_id) / NULLIF(COUNT(DISTINCT s.tour_id), 0), 1), '%')  AS submission_rate
FROM starts s
LEFT JOIN ai_path    ai  ON s.tour_id = ai.tour_id
LEFT JOIN submissions sub ON s.tour_id = sub.tour_id
GROUP BY CASE WHEN ai.tour_id IS NOT NULL THEN 'AI Content Creator' ELSE 'Manual (No AI)' END
ORDER BY submission_rate DESC
""")

display(df_q2_1)

In [ ]:
# Table 2.2 — AI Adoption Rate YoY (Jan–Apr 2025 vs Jan–Apr 2026)
df_q2_2 = spark.sql(f"""
WITH starts AS (
  SELECT
    tour_id,
    YEAR(first_timestamp) AS yr
  FROM production.supply_analytics.tour_creation_funnel
  WHERE step_name = 'Activity Creation Start'
    AND (
      (first_timestamp >= '{BASELINE_START}' AND first_timestamp < '{BASELINE_END}')
      OR (first_timestamp >= '{ANALYSIS_START}' AND first_timestamp < '{ANALYSIS_END}')
    )
),
ai_path AS (
  SELECT DISTINCT tour_id
  FROM production.supply_analytics.tour_creation_funnel
  WHERE step_name = 'AI Content Creator'
)
SELECT
  s.yr                                                                                                            AS year,
  COUNT(DISTINCT s.tour_id)                                                                                      AS tours_started,
  COUNT(DISTINCT ai.tour_id)                                                                                     AS ai_path_starts,
  CONCAT(ROUND(100.0 * COUNT(DISTINCT ai.tour_id) / NULLIF(COUNT(DISTINCT s.tour_id), 0), 1), '%')             AS ai_adoption_rate
FROM starts s
LEFT JOIN ai_path ai ON s.tour_id = ai.tour_id
GROUP BY s.yr
ORDER BY s.yr
""")

display(df_q2_2)

In [ ]:
# Table 2.3 — Time from Creation Start to Submission by Path (2026)
df_q2_3 = spark.sql(f"""
WITH starts AS (
  SELECT tour_id, first_timestamp AS start_ts
  FROM production.supply_analytics.tour_creation_funnel
  WHERE step_name = 'Activity Creation Start'
    AND first_timestamp >= '{ANALYSIS_START}'
    AND first_timestamp <  '{ANALYSIS_END}'
),
ai_path AS (
  SELECT DISTINCT tour_id
  FROM production.supply_analytics.tour_creation_funnel
  WHERE step_name = 'AI Content Creator'
),
submissions AS (
  SELECT tour_id, first_timestamp AS submit_ts
  FROM production.supply_analytics.tour_creation_funnel
  WHERE step_name = 'Activity Submitted'
)
SELECT
  CASE WHEN ai.tour_id IS NOT NULL THEN 'AI Content Creator' ELSE 'Manual (No AI)' END  AS creation_path,
  COUNT(DISTINCT sub.tour_id)                                                             AS tours_submitted,
  ROUND(PERCENTILE_APPROX(
    (unix_timestamp(sub.submit_ts) - unix_timestamp(s.start_ts)) / 3600.0, 0.5), 1)     AS median_hours_to_submit,
  ROUND(PERCENTILE_APPROX(
    (unix_timestamp(sub.submit_ts) - unix_timestamp(s.start_ts)) / 3600.0, 0.25), 1)    AS p25_hours,
  ROUND(PERCENTILE_APPROX(
    (unix_timestamp(sub.submit_ts) - unix_timestamp(s.start_ts)) / 3600.0, 0.75), 1)    AS p75_hours
FROM starts s
INNER JOIN submissions sub ON s.tour_id = sub.tour_id AND sub.submit_ts > s.start_ts
LEFT JOIN  ai_path     ai  ON s.tour_id = ai.tour_id
GROUP BY CASE WHEN ai.tour_id IS NOT NULL THEN 'AI Content Creator' ELSE 'Manual (No AI)' END
""")

display(df_q2_3)

---
## Q3: Where are suppliers dropping off in the creation wizard?

In [ ]:
# Table 3.1 — Wizard Funnel (Jan 15 – Apr 14, 2026)
# Each step shows distinct tour_ids that reached it during the window.
# Step counts are not cumulative: a tour is counted at a step only if its first_timestamp for that step falls within the window.
df_q3_1 = spark.sql(f"""
WITH step_counts AS (
  SELECT
    step_name,
    step_number,
    COUNT(DISTINCT tour_id)  AS tours_reached
  FROM production.supply_analytics.tour_creation_funnel
  WHERE first_timestamp >= '{FUNNEL_START}'
    AND first_timestamp <  '{FUNNEL_END}'
  GROUP BY step_name, step_number
),
start_count AS (
  SELECT tours_reached AS total_starts
  FROM step_counts
  WHERE step_name = 'Activity Creation Start'
)
SELECT
  sc.step_name,
  sc.step_number,
  sc.tours_reached,
  CONCAT(ROUND(100.0 * sc.tours_reached / st.total_starts, 1), '%')  AS pct_of_starts
FROM step_counts sc
CROSS JOIN start_count st
ORDER BY sc.step_number
""")

display(df_q3_1)

---
## Q4: How reliable is the AI content generation system?

In [ ]:
# Table 4.1 — AI Content Generation Success by Month (2026)
df_q4_1 = spark.sql(f"""
SELECT
  DATE_FORMAT(update_timestamp, 'MMM yyyy')                                           AS month,
  MONTH(update_timestamp)                                                             AS month_num,
  COUNT(DISTINCT CASE WHEN content_generation_status = 'completed' THEN tour_id END) AS completed_tours,
  COUNT(DISTINCT CASE WHEN content_generation_status = 'failed'    THEN tour_id END) AS failed_tours,
  CONCAT(ROUND(
    100.0 * COUNT(DISTINCT CASE WHEN content_generation_status = 'completed' THEN tour_id END) /
    NULLIF(COUNT(DISTINCT CASE WHEN content_generation_status IN ('completed', 'failed') THEN tour_id END), 0)
  , 1), '%')                                                                          AS success_rate
FROM production.db_mirror_dbz.catalog__tour_generated_content
WHERE update_timestamp >= '{ANALYSIS_START}'
  AND update_timestamp <  '{ANALYSIS_END}'
GROUP BY DATE_FORMAT(update_timestamp, 'MMM yyyy'), MONTH(update_timestamp)
ORDER BY month_num
""")

display(df_q4_1)

In [ ]:
# Table 4.2 — AI Generation Success Rate YoY (Jan–Apr 2025 vs Jan–Apr 2026)
df_q4_2 = spark.sql(f"""
SELECT
  YEAR(update_timestamp)                                                              AS year,
  COUNT(DISTINCT CASE WHEN content_generation_status = 'completed' THEN tour_id END) AS completed_tours,
  COUNT(DISTINCT CASE WHEN content_generation_status = 'failed'    THEN tour_id END) AS failed_tours,
  CONCAT(ROUND(
    100.0 * COUNT(DISTINCT CASE WHEN content_generation_status = 'completed' THEN tour_id END) /
    NULLIF(COUNT(DISTINCT CASE WHEN content_generation_status IN ('completed', 'failed') THEN tour_id END), 0)
  , 1), '%')                                                                          AS success_rate
FROM production.db_mirror_dbz.catalog__tour_generated_content
WHERE (
    (update_timestamp >= '{BASELINE_START}' AND update_timestamp < '{BASELINE_END}')
    OR (update_timestamp >= '{ANALYSIS_START}' AND update_timestamp < '{ANALYSIS_END}')
)
GROUP BY YEAR(update_timestamp)
ORDER BY year
""")

display(df_q4_2)

---
## Q5: How does creation performance vary by category?

In [ ]:
# Table 5.1 — Submission Rate by Category (Jan 1 – Apr 14, 2026)
# Category sourced from catalog__tour_business_category (level1 field), latest row per tour_id.
df_q5_1 = spark.sql(f"""
WITH starts AS (
  SELECT tour_id
  FROM production.supply_analytics.tour_creation_funnel
  WHERE step_name = 'Activity Creation Start'
    AND first_timestamp >= '{ANALYSIS_START}'
    AND first_timestamp <  '{ANALYSIS_END}'
),
submissions AS (
  SELECT DISTINCT tour_id
  FROM production.supply_analytics.tour_creation_funnel
  WHERE step_name = 'Activity Submitted'
),
categories AS (
  SELECT tour_id, level1 AS category
  FROM production.db_mirror_dbz.catalog__tour_business_category
  QUALIFY ROW_NUMBER() OVER (PARTITION BY tour_id ORDER BY update_timestamp DESC NULLS LAST) = 1
)
SELECT
  COALESCE(c.category, 'Other / Unclassified')                                                                AS category,
  COUNT(DISTINCT s.tour_id)                                                                                   AS tours_started,
  COUNT(DISTINCT sub.tour_id)                                                                                 AS tours_submitted,
  CONCAT(ROUND(100.0 * COUNT(DISTINCT sub.tour_id) / NULLIF(COUNT(DISTINCT s.tour_id), 0), 1), '%')         AS submission_rate
FROM starts s
LEFT JOIN categories  c   ON s.tour_id = c.tour_id
LEFT JOIN submissions sub ON s.tour_id = sub.tour_id
GROUP BY COALESCE(c.category, 'Other / Unclassified')
ORDER BY COUNT(DISTINCT sub.tour_id) / NULLIF(COUNT(DISTINCT s.tour_id), 0) DESC NULLS LAST
""")

display(df_q5_1)

---
## Q6: What share of newly submitted tours receive a first booking within 30 days?

In [ ]:
# Table 6.1 — Activation Rate (1B30D) by Creation Path: January 2025 (baseline) vs January 2026
# Activation: has_booking_in_preceding_28_days = true at submitted_date + 30 days.
# January cohorts used so the full 30-day observation window has elapsed for both years.
df_q6_1 = spark.sql(f"""
WITH jan_submissions AS (
  SELECT
    tour_id,
    DATE(first_timestamp)          AS submitted_date,
    YEAR(first_timestamp)          AS yr
  FROM production.supply_analytics.tour_creation_funnel
  WHERE step_name = 'Activity Submitted'
    AND (
      (first_timestamp >= '2025-01-01' AND first_timestamp < '2025-02-01')
      OR
      (first_timestamp >= '2026-01-01' AND first_timestamp < '2026-02-01')
    )
),
ai_path AS (
  SELECT DISTINCT tour_id
  FROM production.supply_analytics.tour_creation_funnel
  WHERE step_name = 'AI Content Creator'
),
activation AS (
  SELECT tour_id, date AS activation_date, has_booking_in_preceding_28_days
  FROM production.supply.dim_activity_lookback_activation
  WHERE date BETWEEN '2025-01-31' AND '2026-03-05'
)
SELECT
  s.yr                                                                                                                    AS year,
  CASE WHEN ai.tour_id IS NOT NULL THEN 'AI Content Creator' ELSE 'Manual (No AI)' END                                 AS creation_path,
  COUNT(DISTINCT s.tour_id)                                                                                              AS tours_submitted,
  COUNT(DISTINCT CASE WHEN a.has_booking_in_preceding_28_days = TRUE THEN s.tour_id END)                               AS tours_with_booking_30d,
  CONCAT(ROUND(100.0 * COUNT(DISTINCT CASE WHEN a.has_booking_in_preceding_28_days = TRUE THEN s.tour_id END) /
    NULLIF(COUNT(DISTINCT s.tour_id), 0), 1), '%')                                                                      AS activation_rate_1b30d
FROM jan_submissions s
LEFT JOIN ai_path    ai ON s.tour_id = ai.tour_id
LEFT JOIN activation a  ON s.tour_id = a.tour_id AND a.activation_date = DATE_ADD(s.submitted_date, 30)
GROUP BY s.yr, CASE WHEN ai.tour_id IS NOT NULL THEN 'AI Content Creator' ELSE 'Manual (No AI)' END
ORDER BY s.yr, creation_path
""")

display(df_q6_1)

In [ ]:
# Table 6.2 — Overall Activation Rate: January 2025 (baseline) vs January 2026
df_q6_2 = spark.sql(f"""
WITH jan_submissions AS (
  SELECT
    tour_id,
    DATE(first_timestamp)  AS submitted_date,
    YEAR(first_timestamp)  AS yr
  FROM production.supply_analytics.tour_creation_funnel
  WHERE step_name = 'Activity Submitted'
    AND (
      (first_timestamp >= '2025-01-01' AND first_timestamp < '2025-02-01')
      OR
      (first_timestamp >= '2026-01-01' AND first_timestamp < '2026-02-01')
    )
),
activation AS (
  SELECT tour_id, date AS activation_date, has_booking_in_preceding_28_days
  FROM production.supply.dim_activity_lookback_activation
  WHERE date BETWEEN '2025-01-31' AND '2026-03-05'
)
SELECT
  s.yr                                                                                                                    AS year,
  COUNT(DISTINCT s.tour_id)                                                                                              AS tours_submitted,
  COUNT(DISTINCT CASE WHEN a.has_booking_in_preceding_28_days = TRUE THEN s.tour_id END)                               AS tours_with_booking_30d,
  CONCAT(ROUND(100.0 * COUNT(DISTINCT CASE WHEN a.has_booking_in_preceding_28_days = TRUE THEN s.tour_id END) /
    NULLIF(COUNT(DISTINCT s.tour_id), 0), 1), '%')                                                                      AS activation_rate_1b30d
FROM jan_submissions s
LEFT JOIN activation a ON s.tour_id = a.tour_id AND a.activation_date = DATE_ADD(s.submitted_date, 30)
GROUP BY s.yr
ORDER BY s.yr
""")

display(df_q6_2)

---
## Q7: Was there a notable platform issue during the analysis period?

In [ ]:
# Table 7.1 — Weekly Trend: Submission Rate and AI Adoption (Jan–Apr 2026)
df_q7_1 = spark.sql(f"""
WITH starts AS (
  SELECT
    tour_id,
    DATE_TRUNC('week', first_timestamp)           AS week_start,
    DATE_FORMAT(first_timestamp, 'MMM d')         AS week_label
  FROM production.supply_analytics.tour_creation_funnel
  WHERE step_name = 'Activity Creation Start'
    AND first_timestamp >= '{ANALYSIS_START}'
    AND first_timestamp <  '{ANALYSIS_END}'
),
submissions AS (
  SELECT DISTINCT tour_id
  FROM production.supply_analytics.tour_creation_funnel
  WHERE step_name = 'Activity Submitted'
),
ai_path AS (
  SELECT DISTINCT tour_id
  FROM production.supply_analytics.tour_creation_funnel
  WHERE step_name = 'AI Content Creator'
)
SELECT
  s.week_start,
  DATE_FORMAT(s.week_start, 'MMM d')                                                                                  AS week_of,
  COUNT(DISTINCT s.tour_id)                                                                                           AS tours_started,
  COUNT(DISTINCT sub.tour_id)                                                                                         AS tours_submitted,
  CONCAT(ROUND(100.0 * COUNT(DISTINCT sub.tour_id) / NULLIF(COUNT(DISTINCT s.tour_id), 0), 1), '%')                 AS submission_rate,
  CONCAT(ROUND(100.0 * COUNT(DISTINCT ai.tour_id)  / NULLIF(COUNT(DISTINCT s.tour_id), 0), 1), '%')                 AS ai_adoption
FROM starts s
LEFT JOIN submissions sub ON s.tour_id = sub.tour_id
LEFT JOIN ai_path     ai  ON s.tour_id = ai.tour_id
GROUP BY s.week_start, DATE_FORMAT(s.week_start, 'MMM d')
ORDER BY s.week_start
""")

display(df_q7_1)

In [ ]:
# Table 7.2 — Same-Week YoY Comparison: Weekly Submission Rate and AI Adoption (Jan–Apr 2025 vs Jan–Apr 2026)
df_q7_2 = spark.sql(f"""
WITH starts AS (
  SELECT
    tour_id,
    YEAR(first_timestamp)                                 AS yr,
    WEEKOFYEAR(first_timestamp)                           AS week_num,
    DATE_TRUNC('week', first_timestamp)                   AS week_start
  FROM production.supply_analytics.tour_creation_funnel
  WHERE step_name = 'Activity Creation Start'
    AND (
      (first_timestamp >= '{BASELINE_START}' AND first_timestamp < '{BASELINE_END}')
      OR (first_timestamp >= '{ANALYSIS_START}' AND first_timestamp < '{ANALYSIS_END}')
    )
),
submissions AS (
  SELECT DISTINCT tour_id
  FROM production.supply_analytics.tour_creation_funnel
  WHERE step_name = 'Activity Submitted'
),
ai_path AS (
  SELECT DISTINCT tour_id
  FROM production.supply_analytics.tour_creation_funnel
  WHERE step_name = 'AI Content Creator'
)
SELECT
  s.yr                                                                                                                  AS year,
  s.week_num,
  DATE_FORMAT(s.week_start, 'MMM d')                                                                                  AS week_of,
  COUNT(DISTINCT s.tour_id)                                                                                           AS tours_started,
  COUNT(DISTINCT sub.tour_id)                                                                                         AS tours_submitted,
  CONCAT(ROUND(100.0 * COUNT(DISTINCT sub.tour_id) / NULLIF(COUNT(DISTINCT s.tour_id), 0), 1), '%')                 AS submission_rate,
  CONCAT(ROUND(100.0 * COUNT(DISTINCT ai.tour_id)  / NULLIF(COUNT(DISTINCT s.tour_id), 0), 1), '%')                 AS ai_adoption
FROM starts s
LEFT JOIN submissions sub ON s.tour_id = sub.tour_id
LEFT JOIN ai_path     ai  ON s.tour_id = ai.tour_id
GROUP BY s.yr, s.week_num, s.week_start, DATE_FORMAT(s.week_start, 'MMM d')
ORDER BY s.yr, s.week_num
""")

display(df_q7_2)

---
## Appendix: Key Metrics Glossary

| Term | Definition |
|------|-----------|
| **Tour creation start** | A supplier session that reached the `Activity Creation Start` funnel step. Primary denominator for funnel and adoption metrics. |
| **Tour submission** | A creation session that reached the `Activity Submitted` funnel step. The creation wizard is considered completed at this point; the tour enters curation review. |
| **Submission rate** | Share of creation starts that reach submission. Formula: distinct `tour_id` values at `Activity Submitted` / distinct `tour_id` values at `Activity Creation Start` within the same time window. |
| **AI Content Creator path** | A tour creation session that includes the `AI Content Creator` step (step 3 in `tour_creation_funnel`). Triggered when a supplier submits a URL or text description to generate tour content. |
| **Manual path** | A tour creation session that reached `Activity Creation Start` but did not reach the `AI Content Creator` step. |
| **AI adoption rate** | Share of creation starts that also reached the `AI Content Creator` step. Formula: distinct `tour_id` at `AI Content Creator` / distinct `tour_id` at `Activity Creation Start`. |
| **AI content generation success rate** | Share of AI generation requests with `content_generation_status = 'completed'` in `catalog__tour_generated_content`. Denominator is completed + failed tours (unique on `tour_id`). |
| **Activation rate (1B30D)** | Share of submitted tours that received at least one booking within 30 days of submission. Sourced from `dim_activity_lookback_activation` (`has_booking_in_preceding_28_days = true` at `submitted_date + 30 days`). |
| **Category** | Top-level (`level1`) business category from `catalog__tour_business_category`. Assigned post-categorisation; tours without a category match at analysis time appear as "Other / Unclassified". |
| **Funnel step number** | The integer `step_number` field in `tour_creation_funnel`, reflecting the canonical position of each step in the wizard flow. Step numbers are consistent across sessions. |
| **P75 time-to-submit** | The 75th percentile of elapsed hours between `Activity Creation Start` and `Activity Submitted` for tours that completed both steps in sequence (`submit_ts > start_ts`). |
| **2025 baseline** | Jan–Apr 2025 used as the prior-year comparison period for YoY metrics. Same calendar window as the 2026 analysis period. |